# CFRI candle geometry v1 — exploratory matched comparison

This notebook compares a new candle-geometry price imputer with the frozen CFRI
reference on **matched data and masks**. Its design follows inspection of the
2026 results, so this run is exploratory; it is not a prospective confirmation.

**Run all** mounts Google Drive and saves every completed condition under
`MyDrive/ITRI/CFRI_candle_v1_exploratory`. After a Colab interruption, run all
again: source, data, settings, masks and condition checkpoints are verified;
completed conditions are skipped. Changing the embedded code requires a new
Drive results folder. The final results ZIP is downloaded automatically and
excludes raw Yahoo OHLCV files.

If the complete previous `CFRI_holdout_v1_local` results and private data are
still in Drive, the reference stage can reuse them after verification, and a
CPU runtime is enough for the new method. Otherwise it reacquires data and
reruns CFRI and SAITS on the same snapshot: choose **A100 GPU** before Run all.
Standard RAM is normally sufficient; SAITS on a smaller GPU is much slower.
The reference choice is printed after the preparation stage.


In [ ]:
import subprocess, sys
requirements = ['numpy==2.1.3', 'pandas==2.2.3', 'scikit-learn==1.7.2', 'yfinance==1.7.0', 'exchange-calendars==4.13.2', 'pypots==1.5']
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *requirements])
print("Pinned packages installed; PyTorch comes from the selected Colab runtime.")


In [ ]:
import base64, hashlib, io, json, os, zipfile
from datetime import datetime, timezone
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
run_root = Path("/content/drive/MyDrive/ITRI/CFRI_candle_v1_exploratory")
old_root = Path("/content/drive/MyDrive/ITRI/CFRI_holdout_v1_local")
code_dir = Path("/content/cfri_candle_v1_code")
run_root.mkdir(parents=True, exist_ok=True)
code_dir.mkdir(parents=True, exist_ok=True)

embedded_source_sha256 = "38081f50821e2bda2ea2323ce67543f828f0100f264b9027e5a26ad1ef200215"
packed = base64.b64decode(
    "..."
)
if hashlib.sha256(packed).hexdigest() != embedded_source_sha256:
    raise RuntimeError("Embedded code bundle SHA-256 mismatch")

source_path = run_root / "candle_source_bundle.zip"
if source_path.exists():
    if hashlib.sha256(source_path.read_bytes()).hexdigest() != embedded_source_sha256:
        raise RuntimeError("This Drive root has different candle code; restore its original notebook or use a new root")
else:
    temp = source_path.with_name(source_path.name + ".tmp")
    with temp.open("wb") as stream:
        stream.write(packed)
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(temp, source_path)

with zipfile.ZipFile(io.BytesIO(packed)) as archive:
    names = archive.namelist()
    if len(names) != len(set(names)) or archive.testzip() is not None:
        raise RuntimeError("Embedded source ZIP is corrupt or has duplicate paths")
    if any(name.startswith("/") or ".." in Path(name).parts for name in names):
        raise RuntimeError("Embedded source ZIP has an unsafe path")
    expected = json.loads(archive.read("CANDLE_SOURCE_SHA256.json"))
    if set(names) != set(expected) | {"CANDLE_SOURCE_SHA256.json"}:
        raise RuntimeError("Embedded source ZIP and manifest disagree")
    archive.extractall(code_dir)
for name, digest in expected.items():
    path = code_dir / name
    if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != digest:
        raise RuntimeError(f"Source hash mismatch: {name}")

receipt_path = run_root / "candle_source_receipt.json"
if receipt_path.exists():
    receipt = json.loads(receipt_path.read_text())
    if receipt.get("source_sha256") != embedded_source_sha256:
        raise RuntimeError("Drive source receipt does not match this notebook")
else:
    receipt = {"classification": "exploratory after 2026 results",
               "source_sha256": embedded_source_sha256,
               "created_utc": datetime.now(timezone.utc).isoformat()}
    temp = receipt_path.with_name(receipt_path.name + ".tmp")
    temp.write_text(json.dumps(receipt, sort_keys=True, indent=2) + "\n")
    os.replace(temp, receipt_path)
print("Drive results:", run_root)
print("Source bundle SHA-256:", embedded_source_sha256)
print("Source files verified:", len(expected))


In [ ]:
import subprocess, sys
from collections import deque
from datetime import datetime, timezone

def run_stage(name, script, arguments=()):
    command = [sys.executable, "-u", str(code_dir / script), "--root", str(run_root), *arguments]
    recent = deque(maxlen=100)
    log_path = run_root / f"{name}.log"
    with log_path.open("a", encoding="utf-8", buffering=1) as log:
        log.write(f"\n=== {name} started {datetime.now(timezone.utc).isoformat()} UTC ===\n")
        log.write("Command: " + " ".join(command) + "\n")
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              cwd=str(code_dir), text=True, errors="replace", bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                    recent.append(line)
                status = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
        log.write(f"=== {name} ended {datetime.now(timezone.utc).isoformat()} UTC; exit {status} ===\n")
    if status:
        detail = "".join(recent)
        error_path = run_root / f"{name}_last_error.log"
        temp = error_path.with_name(error_path.name + ".tmp")
        temp.write_text(detail, encoding="utf-8")
        os.replace(temp, error_path)
        raise RuntimeError(f"{name} stopped (exit {status}). Last output:\n{detail}")
    print(f"{name}: complete; log saved to {log_path}")


In [ ]:
run_stage("reference_prepare", "reference_stage.py",
          ["--old-root", str(old_root), "--stage", "prepare"])
choice_path = run_root / "reference_choice.json"
if not choice_path.is_file():
    raise RuntimeError("Reference preparation did not create reference_choice.json")
choice = json.loads(choice_path.read_text())
mode = choice.get("mode")
if mode not in ("reuse", "fresh"):
    raise RuntimeError(f"Unexpected reference choice: {choice}")
print("Reference mode:", mode)
if mode == "reuse":
    print("Verified earlier reference scores are available; a CPU runtime is sufficient.")
else:
    import torch
    print("Fresh matched CFRI and SAITS references are required.")
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
    if not torch.cuda.is_available():
        raise RuntimeError("Select an A100 GPU runtime and rerun all cells; fresh SAITS requires CUDA")


In [ ]:
run_stage("reference_cpu", "reference_stage.py", ["--old-root", str(old_root), "--stage", "cpu"])


In [ ]:
run_stage("reference_saits", "reference_stage.py", ["--old-root", str(old_root), "--stage", "saits"])


In [ ]:
run_stage("reference_verify", "reference_stage.py", ["--old-root", str(old_root), "--stage", "verify"])


In [ ]:
run_stage("candle_cpu", "candle_runner.py", ["--stage", "run"])


In [ ]:
from google.colab import files
run_stage("candle_analysis", "candle_analyze.py")
result_zip = run_root / "CFRI_candle_v1_results.zip"
if not result_zip.is_file():
    raise FileNotFoundError(result_zip)
with zipfile.ZipFile(result_zip) as archive:
    if archive.testzip() is not None:
        raise RuntimeError("Final results ZIP failed CRC verification")
    if any("raw_do_not_publish" in name or "raw" in Path(name).parts for name in archive.namelist()):
        raise RuntimeError("Final results ZIP contains a raw-data path")
print("Completed ZIP:", result_zip)
print("ZIP SHA-256:", hashlib.sha256(result_zip.read_bytes()).hexdigest())
files.download(str(result_zip))
